In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_E.xpt"
csv_file = DATA_DIR / "HDL_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 HDL_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.shape != (8132, 3):

    raise ValueError(
        f"Unexpected HDL_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HDL_E structure = "
    "8,132 rows x 3 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# SEQN:
# Identifier -> clean integer formatting
#
# LBDHDD:
# Direct HDL-Cholesterol mg/dL
#
# LBDHDDSI:
# Direct HDL-Cholesterol mmol/L
#
# BOTH cholesterol variables are LAB MEASUREMENTS.
#
# DO NOT:
#
# - convert either measurement to Int64
# - round them
# - remove legitimate .0
# - recalculate LBDHDDSI
# - use float_format
#

INTEGER_COLUMNS = [
    "SEQN"
]


MEASUREMENT_COLUMNS = [
    "LBDHDD",
    "LBDHDDSI"
]


print(
    "\nIdentifier fields:",
    INTEGER_COLUMNS
)

print(
    "Lab measurement fields preserved AS IS:",
    MEASUREMENT_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace it with zero."
    )


print(
    "PASS: HDL_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "Unexpected source zero(s) found."
    )


print(
    "PASS: Original HDL_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. CDC CHECK - LBDHDD
# ============================================================

#
# CDC:
#
# Direct HDL-Cholesterol (mg/dL)
#
# Valid   = 7,387
# Missing = 745
# Range   = 7-155
#
# IMPORTANT:
#
# Values happen to be whole-number results,
# but this is still a LAB MEASUREMENT.
#
# Preserve:
#
# 55.0 -> 55.0
#
# NOT:
#
# 55.0 -> 55
#

s = df["LBDHDD"]


if int(
    s.notna().sum()
) != 7387:

    raise ValueError(
        "LBDHDD valid count differs."
    )


if int(
    s.isna().sum()
) != 745:

    raise ValueError(
        "LBDHDD missing count differs."
    )


if s.min() != 7:

    raise ValueError(
        f"Unexpected LBDHDD minimum: {s.min()}"
    )


if s.max() != 155:

    raise ValueError(
        f"Unexpected LBDHDD maximum: {s.max()}"
    )


print(
    "\nPASS: LBDHDD"
)

print(
    "      Valid   = 7,387"
)

print(
    "      Missing = 745"
)

print(
    "      Range   = 7-155 mg/dL"
)


# ============================================================
# 9. CDC CHECK - LBDHDDSI
# ============================================================

#
# Direct HDL-Cholesterol (mmol/L)
#
# CDC:
#
# Valid   = 7,387
# Missing = 745
# Range   = 0.18-4.01
#
# CDC identifies this as a derived conversion field.
#
# DO NOT recompute it.
# Preserve the released XPT value.
#

s = df["LBDHDDSI"]


if int(
    s.notna().sum()
) != 7387:

    raise ValueError(
        "LBDHDDSI valid count differs."
    )


if int(
    s.isna().sum()
) != 745:

    raise ValueError(
        "LBDHDDSI missing count differs."
    )


if not np.isclose(
    s.min(),
    0.18,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected LBDHDDSI minimum: {s.min()}"
    )


if not np.isclose(
    s.max(),
    4.01,
    rtol=0,
    atol=0
):

    raise ValueError(
        f"Unexpected LBDHDDSI maximum: {s.max()}"
    )


print(
    "PASS: LBDHDDSI"
)

print(
    "      Valid   = 7,387"
)

print(
    "      Missing = 745"
)

print(
    "      Range   = 0.18-4.01 mmol/L"
)


# ============================================================
# 10. FRACTIONAL VALUE CHECK
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "LBDHDDSI": 7384
}


if fractional_columns != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COUNTS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: Genuine fractional-value pattern "
    "matches attached HDL_E XPT."
)


# ============================================================
# 11. LBDHDD WHOLE-NUMBER MEASUREMENT CHECK
# ============================================================

#
# LBDHDD contains no mathematically fractional results.
#
# However:
#
# it is a LAB MEASUREMENT,
# so every valid result must remain float-style.
#

lbdhdd_values = (
    df["LBDHDD"]
    .dropna()
    .to_numpy()
)


if not np.all(
    np.isclose(
        lbdhdd_values,
        np.round(lbdhdd_values),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "Unexpected fractional LBDHDD result."
    )


print(
    "\nPASS: All 7,387 LBDHDD results are "
    "mathematically whole measurements."
)

print(
    "      They will still remain float-style values."
)


# ============================================================
# 12. LEGITIMATE SOURCE .0 COUNTS
# ============================================================

#
# Since measurements remain floats:
#
# LBDHDD:
# every valid value should have source-style .0
#
# LBDHDDSI:
# only three values are mathematically whole:
# 3.0
#

EXPECTED_SOURCE_DOT_ZERO = {
    "LBDHDD": 7387,
    "LBDHDDSI": 3
}


for col, expected_count in (
    EXPECTED_SOURCE_DOT_ZERO.items()
):

    actual_count = sum(
        str(float(value)).endswith(".0")
        for value in df[col].dropna()
    )


    print(
        f"Source {col} values ending '.0': "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected source .0 count."
        )


print(
    "PASS: Legitimate measurement .0 "
    "counts identified."
)


# ============================================================
# 13. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# IMPORTANT:
#
# HDL_E requires NO numeric corrections.
#
# tiny -> zero corrections = 0
#
# Therefore expected_df is exactly the
# original released XPT.
#

expected_df = source_df.copy()


# ============================================================
# 14. CLEAN SEQN ONLY
# ============================================================

#
# SEQN:
#
# 41475.0 -> 41475
#
#
# DO NOT change:
#
# LBDHDD
# LBDHDDSI
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to clean integer formatting."
)

print(
    "PASS: LBDHDD preserved as source-decoded float."
)

print(
    "PASS: LBDHDDSI preserved as source-decoded float."
)


# ============================================================
# 15. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


# SEQN
current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    expected_df["SEQN"].to_numpy(),
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN values changed."
    )


# Measurements
for col in MEASUREMENT_COLUMNS:

    if not np.allclose(
        expected_df[col].to_numpy(),
        df[col].to_numpy(),
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: No underlying numeric value changed."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# recalculation
# decimal quantization
#
# Missing XPT values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 17. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 18. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 19. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 3 columns numerically match "
    "the original XPT exactly."
)


# ============================================================
# 20. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 21. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 8132:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 22. STRICT RAW CSV FIDELITY
# ============================================================

#
# SEQN:
#
# str(int(source_value))
#
#
# Lab measurements:
#
# str(float(source_value))
#
#
# Examples:
#
# 41475.0 -> "41475"
#
# LBDHDD:
# 55.0 -> "55.0"
#
# LBDHDDSI:
# 1.42 -> "1.42"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )

    for example in raw_examples:
        print(example)


    raise ValueError(
        "HDL_E raw CSV formatting differs."
    )


print(
    "PASS: Every CSV token matches the "
    "expected XPT-derived representation."
)


# ============================================================
# 23. UNWANTED SEQN .0 CHECK
# ============================================================

seqn_dot_zero_count = sum(
    row["SEQN"].endswith(".0")
    for row in raw_rows
    if row["SEQN"] != ""
)


print(
    "\nUnwanted SEQN '.0' values:",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains .0 formatting."
    )


print(
    "PASS: No unwanted .0 in SEQN."
)


# ============================================================
# 24. LEGITIMATE LAB .0 CHECK
# ============================================================

print(
    "\n--- LEGITIMATE LAB .0 CHECK ---"
)


for col, expected_count in (
    EXPECTED_SOURCE_DOT_ZERO.items()
):

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    print(
        f"{col:<10}: {actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate .0 formatting changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate lab .0 values preserved."
)


# ============================================================
# 25. STRICT LAB RAW-MEASUREMENT CHECK
# ============================================================

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "HDL measurement representation changed."
    )


print(
    "PASS: Every HDL measurement preserved AS IS."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 28. FINAL CDC CHECK AFTER EXPORT
# ============================================================

final_hdd = csv_df["LBDHDD"]
final_si = csv_df["LBDHDDSI"]


if not (
    int(final_hdd.notna().sum()) == 7387
    and
    int(final_hdd.isna().sum()) == 745
    and
    final_hdd.min() == 7
    and
    final_hdd.max() == 155
):

    raise ValueError(
        "Final LBDHDD values/counts changed."
    )


if not (
    int(final_si.notna().sum()) == 7387
    and
    int(final_si.isna().sum()) == 745
    and
    np.isclose(
        final_si.min(),
        0.18,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        final_si.max(),
        4.01,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "Final LBDHDDSI values/counts changed."
    )


print(
    "\nPASS: Final HDL ranges/counts "
    "still match CDC."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HDL_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)


print(
    "\nLBDHDD:"
)

print(
    "Valid   = 7,387"
)

print(
    "Missing = 745"
)

print(
    "Range   = 7-155"
)

print(
    "Legitimate .0 values = 7,387"
)


print(
    "\nLBDHDDSI:"
)

print(
    "Valid   = 7,387"
)

print(
    "Missing = 745"
)

print(
    "Range   = 0.18-4.01"
)

print(
    "Fractional values = 7,384"
)

print(
    "Legitimate .0 values = 3"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted SEQN .0:",
    seqn_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HDL_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 HDL_E XPT -> CSV
Rows: 8,132
Columns: 3
PASS: HDL_E structure = 8,132 rows x 3 columns.

Identifier fields: ['SEQN']
Lab measurement fields preserved AS IS: ['LBDHDD', 'LBDHDDSI']

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: HDL_E contains NO tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original HDL_E contains no numeric zeros.

--- SEQN CHECK ---
PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

PASS: LBDHDD
      Valid   = 7,387
      Missing = 745
      Range   = 7-155 mg/dL
PASS: LBDHDDSI
      Valid   = 7,387
      Missing = 745
      Range   = 0.18-4.01 mmol/L

--- FRACTIONAL VALUE CHECK ---
LBDHDDSI  : 7,384
PASS: Genuine fractional-value pattern matches attached HDL_E XPT.

PASS: All 7,387 LBDHDD results are mathematically whole measurements.
      They will still remain float-style values.
Source LBDHDD values ending '.0': 7,387
Source LBDHDDSI values ending '.0': 3
PASS: Legitimate measuremen